首先，导入 PyTorch 及其视觉库 torchvision。
我们编写一段设备检测代码，确保 cuda 环境能够正确调用 GPU 进行加速。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 检测是否支持 CUDA，并设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


与 NumPy 手动划分不同，PyTorch 提供了原生的 DataLoader。
这不仅能极其方便地实现小批量梯度下降，还能在后台使用多线程加速数据加载。
transforms.ToTensor() 会自动将 PIL 图像转换为张量，并将像素值从 0-255 归一化到 0.0-1.0。

In [3]:
# 定义数据转换
transform = transforms.Compose([transforms.ToTensor()])

# 下载并加载训练集和测试集
train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# 定义 DataLoader，设置 batch_size 为 64
batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

Number of training batches: 938
Number of test batches: 157


我们将继承 nn.Module 来构建 FNN。
在前面的 NumPy 实现中，我们手动在输出层应用了 Softmax 并计算交叉熵。
在 PyTorch 中，标准做法是不在网络末端添加 Softmax 层，而是直接输出未激活的 Logits（即 $Z^{[2]}$）。
我们随后使用的 nn.CrossEntropyLoss() 会在内部自动且更稳定地计算 Softmax 和对数损失。

In [4]:
class SimpleFNN(nn.Module):
    def __init__(self):
        super(SimpleFNN, self).__init__()
        # 展平层：将 28x28 的二维张量展平为一维的 784
        self.flatten = nn.Flatten()
        # 隐藏层：784 输入，128 输出
        self.hidden = nn.Linear(28 * 28, 128)
        # 激活函数
        self.relu = nn.ReLU()
        # 输出层：128 输入，10 输出 (对应 0-9 类别)
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.hidden(x)
        x = self.relu(x)
        logits = self.output(x)
        return logits


# 实例化模型并将其移动到 GPU
model = SimpleFNN().to(device)
print(model)

SimpleFNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=128, out_features=10, bias=True)
)


我们使用标准的交叉熵损失，并引入现代深度学习中最常用的优化器之一：Adam。
相比于我们之前手写的普通梯度下降，Adam 能够自适应地调整每个参数的学习率，通常收敛速度更快。

In [5]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
# 学习率设置为 0.001 是 Adam 的良好默认值
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

print("Starting training on", device, "...")
for epoch in range(epochs):
    model.train()  # 将模型设置为训练模式
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        # 将数据和标签也移动到 GPU
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. 梯度清零 (极其重要，否则梯度会累加)
        optimizer.zero_grad()

        # 2. 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # 3. 反向传播
        loss.backward()

        # 4. 参数更新
        optimizer.step()

        # 统计指标
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(
        f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%"
    )

Starting training on cuda ...
Epoch [1/10] | Loss: 0.3433 | Accuracy: 90.64%
Epoch [2/10] | Loss: 0.1570 | Accuracy: 95.41%
Epoch [3/10] | Loss: 0.1094 | Accuracy: 96.74%
Epoch [4/10] | Loss: 0.0826 | Accuracy: 97.51%
Epoch [5/10] | Loss: 0.0663 | Accuracy: 98.00%
Epoch [6/10] | Loss: 0.0535 | Accuracy: 98.37%
Epoch [7/10] | Loss: 0.0424 | Accuracy: 98.82%
Epoch [8/10] | Loss: 0.0356 | Accuracy: 98.97%
Epoch [9/10] | Loss: 0.0289 | Accuracy: 99.17%
Epoch [10/10] | Loss: 0.0243 | Accuracy: 99.32%


In [6]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy on 10000 images: {100 * correct / total:.2f}%")

Test Accuracy on 10000 images: 97.79%
